# Composicao das equipes eMulti (CNES) - Base de Dados completa

Reconstroi **quem compoe cada equipe Multiprofissional (eMulti, TP_EQUIPE = 72)** a partir da Base de Dados completa do CNES.

**Estrutura real do CNES (importante):** a equipe e identificada pela chave composta `CO_MUNICIPIO + CO_AREA + SEQ_EQUIPE`, nao por uma coluna INE unica. O INE fica na coluna `CO_EQUIPE` da `tbEquipe`.

Tabelas usadas:
- `tbEquipe` -> TP_EQUIPE, CO_SUB_TIPO_EQUIPE, CO_EQUIPE (INE), CO_UNIDADE, por chave (CO_MUNICIPIO, CO_AREA, SEQ_EQUIPE)
- `rlEstabEquipeProf` -> vinculo profissional x equipe (mesma chave) + CO_PROFISSIONAL_SUS, CO_CBO, DT_ENTRADA, DT_DESLIGAMENTO
- `tbCargaHorariaSus` -> carga horaria por (CO_UNIDADE, CO_PROFISSIONAL_SUS, CO_CBO)
- `tbDadosProfissionalSus` -> CNS e nome do profissional
- `tbAtividadeProfissional` -> descricao oficial do CBO
- `tbEstabelecimento` -> nome fantasia da unidade
- `tbTipoEquipe` / `tbSubTipoEquipe` -> descricoes do tipo e subtipo

**Saidas:** base detalhada e tabelas por categoria, por equipe e categoria x municipio (Excel + CSV).


In [ ]:
# 1) Dependencias (nao usa PySUS; nao mexe no numpy do Colab)
!pip install -q openpyxl
import os, re, glob, zipfile, unicodedata, warnings
import pandas as pd
warnings.filterwarnings("ignore")
print("pandas:", pd.__version__, "| pronto.")


In [ ]:
# 2) Parametros
UF_FILTRO   = "33"     # prefixo IBGE do estado (RJ = "33"); use "" para Brasil inteiro
TIPO_EQUIPE = "72"     # 72 = eMulti
SO_ATIVOS   = True     # mantem so profissionais sem DT_DESLIGAMENTO e equipes sem DT_DESATIVACAO
ENRIQUECER_NOME_MUNICIPIO = True
print("UF:", UF_FILTRO or "Brasil", "| Tipo:", TIPO_EQUIPE, "| So ativos:", SO_ATIVOS)


In [ ]:
# 3) Download automatico da Base de Dados do CNES (FTP do DATASUS)
import ftplib, socket
ANO, MES = 2026, 5
HOST, DIRR = "ftp.datasus.gov.br", "/cnes"
NOME = "BASE_DE_DADOS_CNES_{aaaamm}.ZIP"
TIMEOUT = 30; socket.setdefaulttimeout(120)

def _conectar():
    f = ftplib.FTP(HOST, timeout=TIMEOUT); f.login(); f.set_pasv(True); f.cwd(DIRR); f.voidcmd("TYPE I"); return f
def _existe(f, nome):
    try: return f.size(nome)
    except Exception: return None
def _baixar(f, nome, destino, tam):
    feito = [0]
    try:
        with open(destino, "wb") as out:
            def w(b):
                out.write(b); feito[0] += len(b)
                print(f"\r  baixando: {feito[0]/1e6:8.1f} / {tam/1e6:8.1f} MB", end="")
            f.retrbinary(f"RETR {nome}", w, blocksize=1 << 20)
    except Exception as e:
        print(f"\n  aviso ao finalizar ({type(e).__name__}); conferindo...")
    print()
    real = os.path.getsize(destino) if os.path.exists(destino) else 0
    if real >= tam * 0.999: return destino
    raise IOError(f"download incompleto: {real/1e6:.1f}/{tam/1e6:.1f} MB")

ZIP_PATH = None
try:
    ftp = _conectar(); print("Conectado ao FTP do DATASUS.")
    a, m = ANO, MES
    for _ in range(14):
        nome = NOME.format(aaaamm=f"{a}{m:02d}"); tam = _existe(ftp, nome)
        if tam:
            print(f"Competencia {m:02d}/{a} ({tam/1e6:.0f} MB). Baixando..."); ZIP_PATH = _baixar(ftp, nome, "base_cnes.zip", tam); break
        print(f"  {nome}: indisponivel"); m -= 1
        if m == 0: a -= 1; m = 12
    try: ftp.quit()
    except Exception: pass
except Exception as e:
    print("Falha no FTP:", type(e).__name__, str(e)[:160])
print("ZIP_PATH =", ZIP_PATH or "(use a celula 3b ou ZIP_PATH='base_cnes.zip' se ja baixou)")


In [ ]:
# 3b) Fallback: usar arquivo ja baixado ou subir manualmente
# ZIP_PATH = "base_cnes.zip"
# from google.colab import files
# up = files.upload(); ZIP_PATH = next(iter(up))


In [ ]:
# 4) Descompactar
DEST = "/content/cnes_base"; os.makedirs(DEST, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z: z.extractall(DEST)
csvs = glob.glob(os.path.join(DEST, "**", "*.[cC][sS][vV]"), recursive=True)
print(len(csvs), "CSVs extraidos.")


In [ ]:
# 5) Leitura robusta (sep ; / latin-1) e localizacao de tabela pelo nome base
def ler(nome_base, usecols=None):
    """Le a tabela cujo arquivo e <nome_base><competencia>.csv."""
    alvo = None
    for p in csvs:
        if re.match(rf"{nome_base}\d*\.csv$", os.path.basename(p), re.I):
            alvo = p; break
    if not alvo:
        print(f"  AVISO: tabela {nome_base} nao encontrada"); return pd.DataFrame()
    for sep in [";", ","]:
        for enc in ["latin-1", "utf-8"]:
            try:
                df = pd.read_csv(alvo, sep=sep, encoding=enc, dtype=str,
                                 low_memory=False, on_bad_lines="skip",
                                 usecols=usecols if usecols else None)
                if df.shape[1] >= 1: return df
            except ValueError:
                # usecols com coluna ausente: le tudo
                try:
                    return pd.read_csv(alvo, sep=sep, encoding=enc, dtype=str,
                                       low_memory=False, on_bad_lines="skip")
                except Exception: continue
            except Exception: continue
    return pd.DataFrame()

def col(df, *cands):
    for c in cands:
        if c in df.columns: return c
    return None


In [ ]:
# 6) Equipes eMulti (TP_EQUIPE = 72) e chave composta
eq = ler("tbEquipe")
K = ["CO_MUNICIPIO", "CO_AREA", "SEQ_EQUIPE"]
for k in K: eq[k] = eq[k].astype(str).str.strip()
eq["CHAVE"] = eq[K[0]] + "|" + eq[K[1]] + "|" + eq[K[2]]
eq["TP_EQUIPE"] = eq["TP_EQUIPE"].astype(str).str.strip().str.lstrip("0").replace("", "0")
alvo = TIPO_EQUIPE.lstrip("0") or "0"
eq72 = eq[eq["TP_EQUIPE"] == alvo].copy()
if UF_FILTRO:
    eq72 = eq72[eq72["CO_MUNICIPIO"].str[:2] == UF_FILTRO]
if SO_ATIVOS and "DT_DESATIVACAO" in eq72.columns:
    eq72 = eq72[eq72["DT_DESATIVACAO"].fillna("").str.strip() == ""]

ine_col = col(eq72, "CO_EQUIPE")
sub_col = col(eq72, "CO_SUB_TIPO_EQUIPE")
keys72 = set(eq72["CHAVE"])
print(f"Equipes tipo {TIPO_EQUIPE}: {len(eq72):,} | chaves unicas: {len(keys72):,} | INE unicos: {eq72[ine_col].nunique():,}")


In [ ]:
# 7) Vinculo profissional x equipe (mesma chave composta)
vinc = ler("rlEstabEquipeProf")
for k in K: vinc[k] = vinc[k].astype(str).str.strip()
vinc["CHAVE"] = vinc[K[0]] + "|" + vinc[K[1]] + "|" + vinc[K[2]]
prof = vinc[vinc["CHAVE"].isin(keys72)].copy()
if SO_ATIVOS and "DT_DESLIGAMENTO" in prof.columns:
    prof = prof[prof["DT_DESLIGAMENTO"].fillna("").str.strip() == ""]
prof["CO_PROFISSIONAL_SUS"] = prof["CO_PROFISSIONAL_SUS"].astype(str).str.strip()
prof["CO_CBO"] = prof["CO_CBO"].astype(str).str.strip()
prof["CO_UNIDADE"] = prof["CO_UNIDADE"].astype(str).str.strip()
print(f"Vinculos em eMulti{' (ativos)' if SO_ATIVOS else ''}: {len(prof):,}")


In [ ]:
# 8) Enriquecer: CBO oficial, categoria ampla, carga horaria, CNS, unidade, equipe
# 8.1 descricao oficial do CBO
cbo = ler("tbAtividadeProfissional")
map_cbo = {}
if not cbo.empty:
    cc = col(cbo, "CO_CBO"); cd = col(cbo, "DS_ATIVIDADE_PROFISSIONAL")
    if cc and cd:
        map_cbo = dict(zip(cbo[cc].astype(str).str.strip(), cbo[cd].astype(str).str.strip()))
prof["CBO_DESCRICAO"] = prof["CO_CBO"].map(map_cbo).fillna("CBO " + prof["CO_CBO"])

# 8.2 categoria ampla (familia de 4 digitos) p/ agrupar
FAM = {"2515":"Psicologo","2236":"Fisioterapeuta","2237":"Nutricionista","2238":"Fonoaudiologo",
       "2516":"Assistente Social","2234":"Farmaceutico","2239":"Terapeuta Ocupacional",
       "2241":"Profissional de Educacao Fisica","2235":"Enfermeiro","2232":"Cirurgiao-Dentista",
       "2394":"Psicopedagogo/Pedagogo","3222":"Tecnico/Aux Enfermagem","3251":"Tecnico Saude Bucal",
       "5151":"Agente Comunitario de Saude","2263":"Praticas Integrativas","2212":"Medico (area)"}
MEDP = ("2231","2251","2252","2253")
def categoria(c):
    s = re.sub(r"\D","",str(c)).zfill(6); f = s[:4]
    if f in FAM: return FAM[f]
    if f.startswith(MEDP) or f.startswith("225"): return "Medico"
    return "Outros"
prof["CATEGORIA"] = prof["CO_CBO"].map(categoria)

# 8.3 carga horaria (soma das 3 colunas), por (CO_UNIDADE, CO_PROFISSIONAL_SUS, CO_CBO)
ch = ler("tbCargaHorariaSus",
         usecols=["CO_UNIDADE","CO_PROFISSIONAL_SUS","CO_CBO",
                  "QT_CARGA_HORARIA_AMBULATORIAL","QT_CARGA_HORARIA_OUTROS","QT_CARGA_HOR_HOSP_SUS"])
if not ch.empty:
    for c in ["CO_UNIDADE","CO_PROFISSIONAL_SUS","CO_CBO"]:
        ch[c] = ch[c].astype(str).str.strip()
    qt = [c for c in ["QT_CARGA_HORARIA_AMBULATORIAL","QT_CARGA_HORARIA_OUTROS","QT_CARGA_HOR_HOSP_SUS"] if c in ch.columns]
    ch["CH"] = ch[qt].apply(lambda s: pd.to_numeric(s, errors="coerce")).sum(axis=1)
    ch = ch.groupby(["CO_UNIDADE","CO_PROFISSIONAL_SUS","CO_CBO"], as_index=False)["CH"].sum()
    prof = prof.merge(ch, on=["CO_UNIDADE","CO_PROFISSIONAL_SUS","CO_CBO"], how="left")
else:
    prof["CH"] = pd.NA

# 8.4 CNS / nome
dp = ler("tbDadosProfissionalSus")
if not dp.empty:
    pid = col(dp,"CO_PROFISSIONAL_SUS"); cns = col(dp,"CO_CNS")
    if pid and cns:
        m = dict(zip(dp[pid].astype(str).str.strip(), dp[cns].astype(str).str.strip()))
        prof["CNS"] = prof["CO_PROFISSIONAL_SUS"].map(m).fillna("")
else:
    prof["CNS"] = ""

# 8.5 INE + subtipo da equipe (via eq72)
cols_eq = ["CHAVE", ine_col] + ([sub_col] if sub_col else [])
prof = prof.merge(eq72[cols_eq].drop_duplicates("CHAVE"), on="CHAVE", how="left")
prof = prof.rename(columns={ine_col: "INE"})

# 8.6 subtipo descricao + unidade
sub = ler("tbSubTipoEquipe"); map_sub = {}
if not sub.empty and sub_col:
    sc = col(sub,"CO_SUB_TIPO_EQUIPE"); sd = col(sub,"DS_SUB_TIPO_EQUIPE")
    if sc and sd: map_sub = dict(zip(sub[sc].astype(str).str.strip(), sub[sd].astype(str).str.strip()))
prof["SUBTIPO"] = prof[sub_col].astype(str).str.strip().map(map_sub).fillna("") if sub_col else ""

est = ler("tbEstabelecimento"); map_un = {}; map_cnes = {}
if not est.empty:
    cu = col(est,"CO_UNIDADE"); cf = col(est,"NO_FANTASIA"); cn = col(est,"CO_CNES")
    if cu and cf: map_un = dict(zip(est[cu].astype(str).str.strip(), est[cf].astype(str).str.strip()))
    if cu and cn: map_cnes = dict(zip(est[cu].astype(str).str.strip(), est[cn].astype(str).str.strip()))
prof["UNIDADE"] = prof["CO_UNIDADE"].map(map_un).fillna("")
prof["CNES"] = prof["CO_UNIDADE"].map(map_cnes).fillna(prof["CO_UNIDADE"])
print("Enriquecimento ok.")


In [ ]:
# 9) Municipio (nome) e base detalhada
map_mun = {}
if ENRIQUECER_NOME_MUNICIPIO:
    try:
        import requests
        js = requests.get("https://servicodados.ibge.gov.br/api/v1/localidades/municipios", timeout=30).json()
        for mm in js: map_mun[str(mm["id"])[:6]] = mm["nome"]
    except Exception as e:
        print("Sem IBGE:", e)
prof["COD_MUNICIPIO"] = prof["CO_MUNICIPIO"].astype(str).str.strip()
prof["MUNICIPIO"] = prof["COD_MUNICIPIO"].str[:6].map(map_mun).fillna("")

base_det = prof[["CNES","UNIDADE","COD_MUNICIPIO","MUNICIPIO","INE","SUBTIPO",
                 "CBO_DESCRICAO","CATEGORIA","CO_CBO","CH","CNS"]].copy()
base_det = base_det.rename(columns={"CO_CBO": "CBO"})
base_det = base_det.sort_values(["MUNICIPIO","CNES","INE","CATEGORIA"]).reset_index(drop=True)
n_eq = base_det["INE"].nunique()
print(f"Base detalhada: {len(base_det):,} linhas | eMulti com profissional: {n_eq:,} | "
      f"media prof/equipe: {len(base_det)/max(n_eq,1):.1f}")
base_det.head(15)


In [ ]:
# 10) Tabelas de composicao
comp_categoria = (base_det.groupby("CATEGORIA")
                  .agg(N_profissionais=("CNS","count"), N_equipes=("INE","nunique"), CH_total=("CH","sum"))
                  .sort_values("N_profissionais", ascending=False).reset_index())

comp_cbo = (base_det.groupby("CBO_DESCRICAO")
            .agg(N_profissionais=("CNS","count"), N_equipes=("INE","nunique"))
            .sort_values("N_profissionais", ascending=False).reset_index())

comp_equipe = (base_det.groupby(["INE","CNES","UNIDADE","MUNICIPIO","SUBTIPO"])
               .agg(N_profissionais=("CNS","count"), N_categorias=("CATEGORIA","nunique"),
                    CH_total=("CH","sum"),
                    Categorias=("CATEGORIA", lambda s: ", ".join(sorted(set(s)))))
               .reset_index().sort_values("N_profissionais", ascending=False))

comp_mun = base_det.pivot_table(index="MUNICIPIO", columns="CATEGORIA",
                                values="CNS", aggfunc="count", fill_value=0)
comp_mun["TOTAL"] = comp_mun.sum(axis=1)
comp_mun = comp_mun.sort_values("TOTAL", ascending=False).reset_index()

print("Composicao por categoria:")
display(comp_categoria)


In [ ]:
# 11) Exportar Excel + CSV e baixar
sufixo = (UF_FILTRO or "BR")
xlsx = f"composicao_emulti_{sufixo}.xlsx"
with pd.ExcelWriter(xlsx, engine="openpyxl") as w:
    base_det.to_excel(w, sheet_name="base_detalhada", index=False)
    comp_categoria.to_excel(w, sheet_name="por_categoria", index=False)
    comp_cbo.to_excel(w, sheet_name="por_cbo", index=False)
    comp_equipe.to_excel(w, sheet_name="por_equipe", index=False)
    comp_mun.to_excel(w, sheet_name="categoria_x_municipio", index=False)
base_det.to_csv(f"base_detalhada_{sufixo}.csv", index=False, encoding="utf-8-sig")
print("Gerado:", xlsx)
try:
    from google.colab import files
    files.download(xlsx); files.download(f"base_detalhada_{sufixo}.csv")
except Exception:
    print("Fora do Colab: arquivos em", os.getcwd())


## Notas

- **Chave da equipe.** O CNES identifica a equipe por `CO_MUNICIPIO + CO_AREA + SEQ_EQUIPE`; o INE (`CO_EQUIPE`) e atributo da `tbEquipe`. O join profissional x equipe e feito por essa chave tripla, nao pelo INE.
- **Ativos.** `SO_ATIVOS=True` remove profissionais com `DT_DESLIGAMENTO` e equipes com `DT_DESATIVACAO`, dando o retrato corrente. Coloque `False` para incluir historico do mes.
- **CBO.** A aba `por_cbo` traz a descricao oficial (`tbAtividadeProfissional`); a aba `por_categoria` agrupa em familias amplas para leitura rapida.
- **Subtipo.** A `comp_equipe` traz o subtipo (Ampliada/Complementar/Estrategica), util para nao comparar portes diferentes.
- **Sanidade.** O numero de equipes tipo 72 deve se aproximar do seu agregado (RJ ~ 373) na mesma competencia; equipes sem profissional ativo costumam indicar cadastro incompleto.
